# PRISM-EEG — Periodicity-Rhythm Images with Symmetric hemispheric Modelling

A new model for HMS, designed from the ACNS definitions of the six classes (see `prism.py` docstring and figures):

1. **RPS image per bipolar channel**, computed on the GPU:
   - **R** — signal autocorrelation (rhythm)
   - **P** — envelope autocorrelation (periodicity of discharges)
   - **S** — log spectrum 0.5–20 Hz
2. **Symmetric hemispheric encoder:** one shared pretrained CNN for the left and mirrored right chains, fused as (L+R)/2 and |L−R|, plus a midline stream. The output is exactly invariant to hemisphere swap, polarity and amplitude.
3. **Evolution transformer** over 8 time tokens and their differences (seizure evolution).
4. **ACNS-factorised head:** P(Seizure, PD, RDA, Other) × P(lateralized | PD) × P(lateralized | RDA).

**Shared with existing solutions (data handling only):**
- VIPEEGNet's EEG cache, row table, patient split and validation windows, so scores are paired row by row with the archived baseline
- ±5 s window jitter
- two-stage schedule (all rows weighted votes/20, then ≥10-vote rows)
- an ImageNet-pretrained CNN backbone

**Fair reporting:**
- No checkpoint is chosen on validation. The final EMA weights of every fold are used.
- Outputs go to `fair_comparison/prism_<tag>/` (no TTA) and `…_tta/` (time-reversal TTA), in the archive schema.

**Ablation switches:**
- `HEAD='flat'` (plain 6-way softmax)
- `FUSION='concat'` (no symmetric fusion)
- `REPR='spec'` (spectrum only, no periodicity maps)
- `LOSS='kl+dm'` (adds a Dirichlet-multinomial vote likelihood in stage 1)

## Parameters

In [ ]:
PROJECT_DIR = '~/hms'
TAG = 'v1'
BACKBONE = 'efficientnet_b0'      # any timm model; e.g. 'convnext_nano', 'efficientnet_b2'
HEAD, FUSION, REPR, LOSS = 'acns', 'sym', 'wave+acf', 'kl'   # REPR: 'wave+acf' (v4) | 'wave' | 'rps+wave' (v3) | 'rps' (v2) | 'spec'
WAVE_WIDTH = 1.0                    # channel multiplier of the waveform encoder
NORM, SPECAUG = 'global', True     # v2: keep cross-channel power; time masking. v1 = 'channel', False
EPOCHS1, EPOCHS2 = 12, 5            # waveform models were still improving at epoch 6 (image models overfit)
LR1, LR2, HEAD_LR_MULT = 3e-4, 5e-5, 3.0
BATCH, WORKERS = 32, 8
FOLDS = [0, 1, 2, 3, 4]
SMOKE_TEST = False
SEED = 2024
PRETRAINED = True                 # ImageNet weights via timm (downloads once)

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd, torch
P = os.path.abspath(os.path.expanduser(PROJECT_DIR))
sys.path.insert(0, P)
from prism_train import load_tables, load_eegs, run_fold, kld_rows, TARGETS
from prism import CLASSES
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
torch.backends.cudnn.benchmark = True
if SMOKE_TEST:
    EPOCHS1, EPOCHS2, FOLDS = 2, 1, [0]
RUN = f'prism_{TAG}' + ('_smoke' if SMOKE_TEST else '')
OUT, WDIR = f'{P}/fair_comparison/{RUN}', f'{P}/prism/output_{RUN}'
for d in (OUT, WDIR): os.makedirs(d, exist_ok=True)
SPLIT = f'{P}/results/vipeegnet_groupkfold_v1/folds/patient_folds_groupkfold.csv'
cfg = dict(backbone=BACKBONE, pretrained=PRETRAINED, head=HEAD, fusion=FUSION, repr=REPR, loss=LOSS, dm_weight=0.5,
           norm=NORM, specaug=SPECAUG, wave_width=WAVE_WIDTH,
           epochs1=EPOCHS1, epochs2=EPOCHS2, lr1=LR1, lr2=LR2, head_lr_mult=HEAD_LR_MULT, wd=0.02, clip=2.0,
           ema=0.998, batch=BATCH, workers=WORKERS, seed=SEED)
print(RUN, json.dumps(cfg))

In [ ]:
rows, lookup = load_tables(f'{P}/hms-data', SPLIT, smoke=SMOKE_TEST)
eegs = load_eegs(f'{P}/eeg_cache_v2', rows.eeg_id.unique())
print(f'{len(rows):,} rows | {rows.patient_id.nunique()} patients | >=10 votes: {(rows.votes >= 10).sum():,}')
for k in range(5):
    r = rows[rows.fold == k]; print(f'  fold {k}: {len(r):,} rows, {(r.votes >= 10).sum():,} with >=10 votes')

## Train + out-of-fold prediction

In [ ]:
results = []
t_all = time.time()
for k in FOLDS:
    t0 = time.time()
    res = run_fold(k, rows, lookup, eegs, cfg, OUT, WDIR, device=DEVICE)
    results.append(res)
    print(f'FOLD {k}: KLD >=10 votes {res["kld_hq"]:.4f} (TTA {res["kld_hq_tta"]:.4f}) | all rows {res["kld_all"]:.4f}'
          f' | {(time.time() - t0) / 60:.1f} min', flush=True)
    json.dump(results, open(f'{WDIR}/results_partial.json', 'w'), indent=1)
print(f'done in {(time.time() - t_all) / 3600:.2f} h')

## Summary (same rows as the VIPEEGNet archive)

In [ ]:
def pooled(d, sub):
    yt, yp = [], []
    for k in FOLDS:
        z = np.load(f'{d}/fold{k}_{sub}.npz'); yt.append(z['yt']); yp.append(z['yp'])
    return float(kld_rows(np.concatenate(yt), np.concatenate(yp)).mean()), np.concatenate(yt), np.concatenate(yp)
summ = {'model': 'PRISM-EEG', 'run': RUN, 'cfg': cfg, 'per_fold': results, 'fair_splits': 'VIPEEGNet patient split'}
print('=' * 66)
for tag in ('', '_tta'):
    hq, yt_h, yp_h = pooled(OUT + tag, 'oof'); al, _, _ = pooled(OUT + tag, 'oof_all')
    mf = float(np.mean([r[f'kld_hq{tag}'] for r in results]))
    summ[f'oof_kld_hq{tag or "_native"}'] = hq; summ[f'mean_fold_kld_hq{tag}'] = mf; summ[f'oof_kld_all{tag}'] = al
    print(f'  PRISM {"+ TTA" if tag else "     "}: pooled >=10 votes {hq:.4f} | mean of folds {mf:.4f} | all rows {al:.4f}')
print('  VIPEEGNet       : pooled >=10 votes 0.2217 | mean of folds 0.2221')
print('  per fold (no TTA):', [round(r['kld_hq'], 4) for r in results])
print('=' * 66)
summ['oof_kld_hq_raw'] = summ['oof_kld_hq_native']; summ['n_oof_hq'] = int(sum(r['n_valid_hq'] for r in results))
summ['recipe'] = f'PRISM {HEAD}/{FUSION}/{REPR}/{LOSS} {BACKBONE} {EPOCHS1}+{EPOCHS2} ep'; summ['tta'] = 'none'
for d in (OUT, OUT + '_tta'):
    json.dump(summ, open(f'{d}/summary.json', 'w'), indent=1, default=float)

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix
_, yt, yp = pooled(OUT, 'oof')
yti, ypi = yt.argmax(1), yp.argmax(1); oh = (yti[:, None] == np.arange(6)).astype(int)
cm = confusion_matrix(yti, ypi, labels=range(6))
with np.errstate(invalid='ignore', divide='ignore'):
    rec, prec = np.diag(cm) / cm.sum(1) * 100, np.diag(cm) / cm.sum(0) * 100
vip = {'AUROC': [0.975, 0.963, 0.976, 0.943, 0.952, 0.929]}
print(f'{"class":>8} | AUROC (VIPEEGNet) | recall | precision')
for c in range(6):
    print(f'{CLASSES[c]:>8} | {roc_auc_score(oh[:, c], yp[:, c]):.3f} ({vip["AUROC"][c]:.3f})   | {rec[c]:5.1f}% | {prec[c]:5.1f}%')